In [4]:
# The standard Spark way to list files in Fabric
files = mssparkutils.fs.ls("Files/")
for file in files:
    print(file.name)

StatementMeta(, a5083e1f-1d92-4401-b876-ace2fa105c0b, 6, Finished, Available, Finished, False)

sample_sales.csv


In [6]:
# ----------------------------------------------------------------------------------
# PROJECT: Sales Insights Pipeline
# ENGINEER: Fahmy M
# STAGE: Silver Layer - Data Cleansing & Schema Enforcement
# ----------------------------------------------------------------------------------

from pyspark.sql.functions import col, to_date

# Load raw dataset from OneLake Bronze layer
raw_sales_df = spark.read.format("csv").option("header","true").load("Files/sample_sales.csv")

# Perform data transformations
# 1. Standardize date format
# 2. Cast financial values to Double
# 3. Handle missing values
refined_sales_df = raw_sales_df.withColumn("Date", to_date(col("Date"))) \
                               .withColumn("Amount", col("Amount").cast("double")) \
                               .dropna()

# Write output to Delta table for high-performance querying
refined_sales_df.write.format("delta").mode("overwrite").saveAsTable("silver_sales_refined")

print("Silver layer transformation complete.")

StatementMeta(, a5083e1f-1d92-4401-b876-ace2fa105c0b, 8, Finished, Available, Finished, False)

Silver layer transformation complete.


In [7]:
# ----------------------------------------------------------------------------------
# STAGE: Gold Layer - Regional Performance Aggregation
# PURPOSE: Summarizing revenue by region for executive reporting
# ----------------------------------------------------------------------------------

# Aggregate revenue totals by geographical region
regional_performance_df = refined_sales_df.groupBy("Region").sum("Amount") \
                                          .withColumnRenamed("sum(Amount)", "Total_Revenue")

# Persist to Gold table
regional_performance_df.write.format("delta").mode("overwrite").saveAsTable("gold_regional_performance")

print("Gold business-ready table created successfully.")

StatementMeta(, a5083e1f-1d92-4401-b876-ace2fa105c0b, 9, Finished, Available, Finished, False)

Gold business-ready table created successfully.
